# 가설 검증: 이탈 방어 캠페인의 골든타임

## 검증 가설
> 고가치 고객의 캠페인 증분 매출 효과는 구매주기가 정상 범위인 단계나 크게 지연된 고위험 단계보다, 평소 구매주기의 1.5~3배가 지난 초기 위험 단계에서 가장 클 것이다.

- H0: 위험 단계에 따른 캠페인 증분 매출 차이가 없다.
- H1: 초기 위험 단계의 증분 매출이 정상·주의 및 고위험 단계보다 모두 크다.

## 사전에 고정한 기준
- 고객가치: 캠페인 시작 전 12주(84일) 구매 빈도와 매출의 복합점수 상위 30%
- 구매주기: 캠페인 전 최대 26주(182일)의 구매일 간격 중앙값
- 정상·주의: 구매주기 대비 지연 1.5배 이하
- 초기 위험: 1.5배 초과~3배 이하
- 고위험: 3배 초과
- 성과: 고객 1명당 일평균 증분 매출
- 분석 대상: 종료일까지 완전히 관측되는 캠페인, 단계별 처리군 최소 10명

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
TRANSACTION_PATH = PROJECT_DIR / "transaction_data.csv"
CAMPAIGN_DESC_PATH = PROJECT_DIR / "campaign_desc.csv"
CAMPAIGN_TABLE_PATH = PROJECT_DIR / "campaign_table.csv"
DATASET_END_DAY = 711
VALUE_LOOKBACK_DAYS = 84
CYCLE_LOOKBACK_DAYS = 182
MIN_TREATED_PER_CELL = 10
RANDOM_SEED = 42

## 1. 데이터 불러오기

In [ ]:
tx = pd.read_csv(
    TRANSACTION_PATH,
    usecols=["household_key","BASKET_ID","DAY","SALES_VALUE"],
    dtype={"household_key":"int32","BASKET_ID":"int64","DAY":"int16","SALES_VALUE":"float32"},
)
campaign_desc = pd.read_csv(CAMPAIGN_DESC_PATH)
campaign_table = pd.read_csv(CAMPAIGN_TABLE_PATH)
all_households = np.sort(tx["household_key"].unique())
valid_campaigns = campaign_desc.loc[campaign_desc["END_DAY"] <= DATASET_END_DAY].copy()
print(f"거래 고객: {len(all_households):,}명")
print(f"완전 관측 캠페인: {len(valid_campaigns)}개")

## 2. 캠페인 시작일 기준 고객 상태 생성

현재 시점의 세그먼트를 과거 캠페인에 붙이지 않고, 각 캠페인 시작일 이전 데이터만 사용합니다. 구매주기를 계산하려면 최근 26주에 최소 3개의 서로 다른 구매일이 있어야 합니다.

In [ ]:
def build_snapshot(start_day):
    value_start = max(1, start_day - VALUE_LOOKBACK_DAYS)
    value_tx = tx.loc[tx["DAY"].between(value_start, start_day - 1)]
    value = value_tx.groupby("household_key").agg(
        value_revenue=("SALES_VALUE","sum"), value_baskets=("BASKET_ID","nunique")
    ).reindex(all_households, fill_value=0)
    active = value["value_baskets"] > 0
    value["revenue_pct"] = 0.0
    value["frequency_pct"] = 0.0
    value.loc[active, "revenue_pct"] = value.loc[active, "value_revenue"].rank(pct=True)
    value.loc[active, "frequency_pct"] = value.loc[active, "value_baskets"].rank(pct=True)
    value["value_score"] = (value["revenue_pct"] + value["frequency_pct"]) / 2
    cutoff = value.loc[active, "value_score"].quantile(.70)
    value["high_value"] = active & (value["value_score"] >= cutoff)

    cycle_start = max(1, start_day - CYCLE_LOOKBACK_DAYS)
    visits = tx.loc[tx["DAY"].between(cycle_start, start_day - 1), ["household_key","DAY"]].drop_duplicates()
    visits = visits.sort_values(["household_key","DAY"])
    visits["gap"] = visits.groupby("household_key")["DAY"].diff()
    cycle = visits.groupby("household_key").agg(
        visit_days=("DAY","nunique"), last_purchase_day=("DAY","max"), median_gap_days=("gap","median")
    )
    snapshot = value.join(cycle, how="left").reset_index()
    snapshot["recency_days"] = start_day - snapshot["last_purchase_day"]
    snapshot["gap_ratio"] = snapshot["recency_days"] / snapshot["median_gap_days"].replace(0,np.nan)
    snapshot.loc[snapshot["visit_days"] < 3, "gap_ratio"] = np.nan
    snapshot["risk_stage"] = pd.cut(
        snapshot["gap_ratio"], [-np.inf,1.5,3,np.inf], labels=["정상·주의","초기 위험","고위험"]
    )
    return snapshot

def outcome_metrics(start_day, end_day):
    duration = end_day - start_day + 1
    pre_start = start_day - duration
    window = tx.loc[tx["DAY"].between(pre_start, end_day)].copy()
    window["period"] = np.where(window["DAY"] < start_day, "pre", "during")
    metrics = window.groupby(["household_key","period"]).agg(
        revenue=("SALES_VALUE","sum"), baskets=("BASKET_ID","nunique")
    ).unstack(fill_value=0)
    metrics.columns = [f"{metric}_{period}" for metric,period in metrics.columns]
    metrics = metrics.reindex(all_households, fill_value=0).reset_index()
    for col in ["revenue_pre","revenue_during","baskets_pre","baskets_during"]:
        if col not in metrics: metrics[col] = 0
        metrics[f"{col}_daily"] = metrics[col] / duration
    return metrics, pre_start

## 3. 같은 위험 단계 안에서 유사 비대상 고객 매칭

캠페인 직전 일평균 매출과 장바구니 수가 가장 가까운 고객을 비교군으로 선택합니다. 같은 분석 기간에 다른 캠페인을 받은 고객은 비교군에서 제외합니다.

In [ ]:
def nearest_controls(treated, controls):
    cols = ["revenue_pre_daily","baskets_pre_daily","value_score"]
    combined = pd.concat([treated[cols], controls[cols]])
    x = np.log1p(combined.to_numpy(float))
    mean, std = x.mean(axis=0), x.std(axis=0)
    std[std == 0] = 1
    t = (np.log1p(treated[cols].to_numpy(float)) - mean) / std
    c = (np.log1p(controls[cols].to_numpy(float)) - mean) / std
    distance = ((t[:,None,:] - c[None,:,:])**2).sum(axis=2)
    return controls.iloc[distance.argmin(axis=1)].reset_index(drop=True)

def smd(a,b):
    pooled = np.sqrt((a.var(ddof=1)+b.var(ddof=1))/2)
    return 0 if pooled == 0 else (a.mean()-b.mean())/pooled

## 4. 캠페인×위험단계별 증분 매출 계산

In [ ]:
stage_results, pair_results = [], []
for campaign in valid_campaigns.sort_values("CAMPAIGN").itertuples(index=False):
    cid, start, end = int(campaign.CAMPAIGN), int(campaign.START_DAY), int(campaign.END_DAY)
    snapshot = build_snapshot(start)
    metrics, pre_start = outcome_metrics(start,end)
    data = snapshot.merge(metrics,on="household_key",how="left")
    treated_ids = set(campaign_table.loc[campaign_table["CAMPAIGN"]==cid,"household_key"])
    overlaps = campaign_desc.loc[(campaign_desc["START_DAY"]<=end)&(campaign_desc["END_DAY"]>=pre_start),"CAMPAIGN"]
    overlap_ids = set(campaign_table.loc[campaign_table["CAMPAIGN"].isin(overlaps),"household_key"])

    for stage in ["정상·주의","초기 위험","고위험"]:
        eligible = data.loc[data["high_value"] & data["risk_stage"].eq(stage)].copy()
        treated = eligible.loc[eligible["household_key"].isin(treated_ids)].reset_index(drop=True)
        controls = eligible.loc[~eligible["household_key"].isin(overlap_ids)].reset_index(drop=True)
        if len(treated)<MIN_TREATED_PER_CELL or len(controls)<MIN_TREATED_PER_CELL:
            continue
        matched = nearest_controls(treated,controls)
        t_change = treated["revenue_during_daily"].to_numpy()-treated["revenue_pre_daily"].to_numpy()
        c_change = matched["revenue_during_daily"].to_numpy()-matched["revenue_pre_daily"].to_numpy()
        effect = t_change-c_change
        se = effect.std(ddof=1)/np.sqrt(len(effect))
        balance = smd(treated["revenue_pre_daily"],matched["revenue_pre_daily"])
        stage_results.append({
            "campaign":cid,"campaign_type":campaign.DESCRIPTION,"risk_stage":stage,
            "treated_customers":len(treated),"control_customers":matched["household_key"].nunique(),
            "pre_revenue_balance_smd":balance,"incremental_revenue_daily":effect.mean(),
            "ci_low":effect.mean()-1.96*se,"ci_high":effect.mean()+1.96*se,
        })
        pair_results.append(pd.DataFrame({
            "campaign":cid,"risk_stage":stage,"treated_household":treated["household_key"],
            "control_household":matched["household_key"],"incremental_revenue_daily":effect
        }))

campaign_stage_effects = pd.DataFrame(stage_results)
golden_timing_pairs = pd.concat(pair_results,ignore_index=True) if pair_results else pd.DataFrame()
campaign_stage_effects["quality_pass"] = campaign_stage_effects["pre_revenue_balance_smd"].abs()<=.1
display(campaign_stage_effects.head())

## 5. 가설 검정

매칭 품질이 양호한 셀만 사용합니다. 같은 캠페인 안에서 초기 위험 단계와 다른 단계의 효과 차이를 계산한 뒤, 캠페인을 단위로 부트스트랩 신뢰구간과 단측 부호순열 p-value를 구합니다.

In [ ]:
def paired_stage_test(effect_table, comparison_stage, n_boot=10000, n_perm=20000):
    pivot = effect_table.pivot(index="campaign",columns="risk_stage",values="incremental_revenue_daily")
    pivot = pivot.reindex(columns=["정상·주의","초기 위험","고위험"])
    paired = pivot[["초기 위험",comparison_stage]].dropna()
    diff = (paired["초기 위험"]-paired[comparison_stage]).to_numpy()
    if len(diff)<2:
        return {"comparison":f"초기 위험 - {comparison_stage}","campaigns":len(diff),"mean_difference":np.nan,"ci_low":np.nan,"ci_high":np.nan,"p_value_one_sided":np.nan}
    rng = np.random.default_rng(RANDOM_SEED)
    boot = rng.choice(diff,size=(n_boot,len(diff)),replace=True).mean(axis=1)
    signs = rng.choice([-1,1],size=(n_perm,len(diff)))
    null_means = (signs*diff).mean(axis=1)
    observed = diff.mean()
    return {
        "comparison":f"초기 위험 - {comparison_stage}","campaigns":len(diff),
        "mean_difference":observed,"ci_low":np.quantile(boot,.025),"ci_high":np.quantile(boot,.975),
        "p_value_one_sided":(1+(null_means>=observed).sum())/(n_perm+1),
    }

qualified = campaign_stage_effects.loc[campaign_stage_effects["quality_pass"]].copy()
hypothesis_tests = pd.DataFrame([
    paired_stage_test(qualified,"정상·주의"), paired_stage_test(qualified,"고위험")
])
display(hypothesis_tests)

## 6. 단계별 요약과 최종 판정

In [ ]:
risk_stage_summary = qualified.groupby("risk_stage").agg(
    campaign_cells=("campaign","nunique"), total_treated=("treated_customers","sum"),
    mean_incremental_revenue_daily=("incremental_revenue_daily","mean"),
    median_incremental_revenue_daily=("incremental_revenue_daily","median"),
    positive_effect_rate=("incremental_revenue_daily",lambda x:(x>0).mean())
).reset_index()
display(risk_stage_summary)

valid_tests = hypothesis_tests.dropna(subset=["p_value_one_sided"])
hypothesis_supported = (len(valid_tests)==2 and (valid_tests["mean_difference"]>0).all() and (valid_tests["p_value_one_sided"]<.05).all())
if hypothesis_supported:
    print("판정: 두 비교 모두 초기 위험 단계가 유의하게 높아 가설을 지지합니다.")
else:
    print("판정: 두 비교를 모두 통과하지 못해 현재 데이터로는 가설을 지지할 충분한 근거가 없습니다.")

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=qualified,x="risk_stage",y="incremental_revenue_daily",order=["정상·주의","초기 위험","고위험"],errorbar=("ci",95))
plt.axhline(0,color="black",linewidth=1)
plt.title("고가치 고객의 위험 단계별 캠페인 증분 매출")
plt.xlabel("캠페인 시작 당시 위험 단계"); plt.ylabel("고객당 일평균 증분 매출")
plt.tight_layout(); plt.show()

## 7. 해석 주의사항

- 가설 지지는 두 비교가 모두 양수이고 단측 p-value가 0.05 미만일 때만 선언합니다.
- 유의하지 않으면 가설이 거짓이라고 증명된 것이 아니라, 현재 데이터로 충분한 근거를 얻지 못한 것입니다.
- 캠페인 배정이 무작위가 아니므로 매칭 후에도 관측되지 않은 고객 차이가 남을 수 있습니다.
- 단계별 처리군 10명 미만인 캠페인 셀은 제외하므로 결과가 일부 큰 캠페인에 의해 좌우될 수 있습니다.

In [ ]:
campaign_stage_effects.to_csv(OUTPUT_DIR/"campaign_golden_timing_effects.csv",index=False,encoding="utf-8-sig")
risk_stage_summary.to_csv(OUTPUT_DIR/"golden_timing_stage_summary.csv",index=False,encoding="utf-8-sig")
hypothesis_tests.to_csv(OUTPUT_DIR/"golden_timing_hypothesis_tests.csv",index=False,encoding="utf-8-sig")
print("가설 검증 결과 저장 완료")

# 추가 탐색 검증: 전체 고객의 캠페인 골든타임

고가치 고객 한정 검증은 고위험 단계의 유효 표본이 없어 검증 불충분으로 끝났습니다. 아래 분석은 그 결과를 확인한 뒤 수정한 **사후 탐색 가설**이므로 최초 가설의 확증 결과와 구분합니다.

> 전체 고객에서 초기 위험 단계의 캠페인 증분 매출 효과는 정상·주의 단계와 고위험 단계보다 클 것이다.

고객가치는 대상 제한에 사용하지 않고 비교군 매칭 변수로 통제합니다. 위험단계, 최소 표본, 매칭 품질, 성과지표 기준은 기존 분석과 동일하게 유지합니다.

In [ ]:
all_stage_results = []
for campaign in valid_campaigns.sort_values("CAMPAIGN").itertuples(index=False):
    cid, start, end = int(campaign.CAMPAIGN), int(campaign.START_DAY), int(campaign.END_DAY)
    snapshot = build_snapshot(start)
    metrics, pre_start = outcome_metrics(start,end)
    data = snapshot.merge(metrics,on="household_key",how="left")
    treated_ids = set(campaign_table.loc[campaign_table["CAMPAIGN"]==cid,"household_key"])
    overlaps = campaign_desc.loc[(campaign_desc["START_DAY"]<=end)&(campaign_desc["END_DAY"]>=pre_start),"CAMPAIGN"]
    overlap_ids = set(campaign_table.loc[campaign_table["CAMPAIGN"].isin(overlaps),"household_key"])

    for stage in ["정상·주의","초기 위험","고위험"]:
        eligible = data.loc[data["risk_stage"].eq(stage)].copy()
        treated = eligible.loc[eligible["household_key"].isin(treated_ids)].reset_index(drop=True)
        controls = eligible.loc[~eligible["household_key"].isin(overlap_ids)].reset_index(drop=True)
        if len(treated)<MIN_TREATED_PER_CELL or len(controls)<MIN_TREATED_PER_CELL:
            continue
        matched = nearest_controls(treated,controls)
        t_change = treated["revenue_during_daily"].to_numpy()-treated["revenue_pre_daily"].to_numpy()
        c_change = matched["revenue_during_daily"].to_numpy()-matched["revenue_pre_daily"].to_numpy()
        effect = t_change-c_change
        se = effect.std(ddof=1)/np.sqrt(len(effect))
        balance = smd(treated["revenue_pre_daily"],matched["revenue_pre_daily"])
        all_stage_results.append({
            "campaign":cid,"campaign_type":campaign.DESCRIPTION,"risk_stage":stage,
            "treated_customers":len(treated),"control_customers":matched["household_key"].nunique(),
            "pre_revenue_balance_smd":balance,"incremental_revenue_daily":effect.mean(),
            "ci_low":effect.mean()-1.96*se,"ci_high":effect.mean()+1.96*se,
        })

campaign_all_stage_effects = pd.DataFrame(all_stage_results)
campaign_all_stage_effects["quality_pass"] = campaign_all_stage_effects["pre_revenue_balance_smd"].abs()<=.1
display(campaign_all_stage_effects.groupby("risk_stage").agg(
    campaign_cells=("campaign","nunique"), quality_cells=("quality_pass","sum"),
    treated_customers=("treated_customers","sum")
))

In [ ]:
qualified_all = campaign_all_stage_effects.loc[campaign_all_stage_effects["quality_pass"]].copy()
all_customer_hypothesis_tests = pd.DataFrame([
    paired_stage_test(qualified_all,"정상·주의"), paired_stage_test(qualified_all,"고위험")
])
all_customer_stage_summary = qualified_all.groupby("risk_stage").agg(
    campaign_cells=("campaign","nunique"), total_treated=("treated_customers","sum"),
    mean_incremental_revenue_daily=("incremental_revenue_daily","mean"),
    median_incremental_revenue_daily=("incremental_revenue_daily","median"),
    positive_effect_rate=("incremental_revenue_daily",lambda x:(x>0).mean())
).reset_index()
display(all_customer_stage_summary)
display(all_customer_hypothesis_tests)

valid_all_tests = all_customer_hypothesis_tests.dropna(subset=["p_value_one_sided"])
all_hypothesis_supported = (len(valid_all_tests)==2 and (valid_all_tests["mean_difference"]>0).all() and (valid_all_tests["p_value_one_sided"]<.05).all())
if all_hypothesis_supported:
    print("탐색 판정: 두 비교 모두 초기 위험 단계가 유의하게 높아 수정 가설을 지지합니다.")
else:
    print("탐색 판정: 두 비교를 모두 통과하지 못해 수정 가설을 지지할 충분한 근거가 없습니다.")

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=qualified_all,x="risk_stage",y="incremental_revenue_daily",order=["정상·주의","초기 위험","고위험"],errorbar=("ci",95))
plt.axhline(0,color="black",linewidth=1)
plt.title("전체 고객의 위험 단계별 캠페인 증분 매출")
plt.xlabel("캠페인 시작 당시 위험 단계"); plt.ylabel("고객당 일평균 증분 매출")
plt.tight_layout(); plt.show()

In [ ]:
campaign_all_stage_effects.to_csv(OUTPUT_DIR/"campaign_golden_timing_all_effects.csv",index=False,encoding="utf-8-sig")
all_customer_stage_summary.to_csv(OUTPUT_DIR/"golden_timing_all_stage_summary.csv",index=False,encoding="utf-8-sig")
all_customer_hypothesis_tests.to_csv(OUTPUT_DIR/"golden_timing_all_hypothesis_tests.csv",index=False,encoding="utf-8-sig")
print("전체 고객 탐색 검증 결과 저장 완료")